# V4 — Restricted public data + model typography calibration

Self-contained Kaggle submission notebook. Local 5-fold GroupKFold OOF Balanced Accuracy = 0.400062 (not a leaderboard score). Uses the official competition data and two public external datasets. Only a selected, provenance-restricted external subset is used; no Kaggle Benchmarks Proxy sources or reconstructed original training prompts. No prompt-group quotas in prediction.

External data credits: [prvsiyan Public in-style pool v1](https://www.kaggle.com/datasets/prvsiyan/ntl-public-in-style-pool-v1), CC BY-SA 4.0; [prvsiyan Extended Name That LLM](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset), CC BY 4.0. Underlying public pool source authors are recorded in the attached dataset and retained in each sample's `source` field.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def find_file(name, preferred=None):
    if preferred is not None and (Path(preferred)/name).is_file():
        return Path(preferred)/name
    options=sorted(Path("/kaggle/input").rglob(name))
    if len(options)!=1:
        raise FileNotFoundError(f"Expected one {name} under Kaggle input, found {options}")
    return options[0]

train=pd.read_csv(find_file("train.csv", "/kaggle/input/competitions/name-that-llm"))
test=pd.read_csv(find_file("test.csv", "/kaggle/input/competitions/name-that-llm"))
assert list(train.columns)==["id","prompt_id","text","model"]
assert list(test.columns)==["id","text"]
assert train.model.nunique()==18
assert train.id.is_unique and test.id.is_unique
assert train.notna().all().all() and test.notna().all().all()
print("train",train.shape,"test",test.shape)


In [ ]:
TOKENS = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?", re.UNICODE)
SENTENCES = re.compile(r"[.!?]+(?:\s|$)")
WORD_SET = [
    "the", "and", "of", "to", "in", "is", "that", "this", "which",
    "with", "as", "by", "for", "on", "from", "are", "can", "may", "might",
    "will", "should", "could", "we", "you", "i", "they", "it", "not",
    "but", "or", "a", "an", "their", "its", "our", "your",
    "however", "therefore", "indeed", "perhaps", "ultimately",
    "although", "while", "because", "rather", "instead", "through",
    "each", "every", "even", "yet", "only", "often", "also",
]
PHRASES = [
    "for example", "such as", "in fact", "rather than",
    "not only", "not just", "on the other hand", "in other words",
    "at the same time", "in this way", "as a result",
]
PUNCT = list(",;:.!?-—–()[]{}\"'“”‘’/\\*#\n")


def style_vector(text: str) -> np.ndarray:
    """Surface-only, topic-agnostic style statistics."""
    words = [w.lower() for w in TOKENS.findall(text)]
    n = max(1, len(words))
    chars = max(1, len(text))
    lower = text.lower()
    count = {}
    for w in words:
        count[w] = count.get(w, 0) + 1
    counts = np.asarray(list(count.values()), dtype=float) if count else np.array([0.0])
    sent_len = [len(TOKENS.findall(s)) for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if not sent_len:
        sent_len = [n]
    words_len = [len(w) for w in words] or [0]
    vals = [
        np.log1p(chars), np.log1p(n),
        float(np.mean(words_len)), float(np.std(words_len)),
        np.mean(sent_len), np.std(sent_len),
        len(count) / n, np.sum(counts == 1) / n,
        np.sum(counts >= 3) / n,
        sum(len(w) >= 8 for w in words) / n,
        sum(len(w) <= 3 for w in words) / n,
        sum(w[0].isupper() for w in TOKENS.findall(text)) / n,
        sum(c.isupper() for c in text) * 100 / chars,
        sum(c.isdigit() for c in text) * 100 / chars,
        sum(ord(c) > 127 for c in text) * 100 / chars,
        sum(c.isspace() for c in text) * 100 / chars,
        text.count("\n\n") * 100 / chars,
        len(re.findall(r"\b\d+(?:\.\d+)?\b", text)) / n,
    ]
    vals.extend([text.count(p) * 100 / chars for p in PUNCT])
    vals.extend([count.get(w, 0) * 100 / n for w in WORD_SET])
    vals.extend([lower.count(p) * 100 / n for p in PHRASES])
    vals.append(sum(text.startswith(prefix) for prefix in (
        "In ", "The ", "A ", "When ", "Imagine ", "Consider ", "There ", "Every ",
    )))
    return np.asarray(vals, dtype=np.float32)


def style_matrix(texts) -> np.ndarray:
    return np.asarray([style_vector(text) for text in texts], dtype=np.float32)

In [ ]:
# Reconstruct the audited, provenance-filtered external corpus deterministically.
GOOD_SOURCES=(
    "willrice/name-that-llm-openrouter-harvest",
    "seyitkaangunes/ntl-extra-api-passages",
    "michaelsemenoff/ntl-own-extra-v1",
    "zbigniewkirchner/ntl-extra-gpt-passages",
    "prvsiyan/ntl-own-prompt-generations-v1",
)
ROUTES=("anthropic","openai","google","openrouter")
pool=pd.read_parquet(find_file("pool_v1.parquet"))
extra=pd.read_csv(find_file("passages_clean.csv"))
a=pool.loc[pool.source.isin(GOOD_SOURCES)]
a=a.loc[a.model.ne("gemini-3.7-flash")]
blocks=[g.sample(n=min(300,len(g)),random_state=20261118) for _,g in a.groupby("model",sort=True)]
p1=pd.concat(blocks,ignore_index=True)[["prompt_id","model","text","source"]].copy()
p1["provenance"]="public_in_style_pool_v1"
p2=extra.loc[
    extra["split"].eq("new")&extra["provider_route"].isin(ROUTES),
    ["prompt_id","model","text","provider_route"]
].copy()
p2["source"]="prvsiyan/prvsiyans-extended-name-that-llm-dataset"
p2["provenance"]="prvsiyan_new_nonproxy"
p2=p2[["prompt_id","model","text","source","provenance"]]
external=pd.concat([p1,p2],ignore_index=True)
external=external.dropna(subset=["model","text"])
external=external.loc[external.model.isin(train.model.unique())].copy()
def normalized(text): return re.sub(r"\s+"," ",str(text).strip().lower())
external["normalized"]=external.text.map(normalized)
assert not external["normalized"].isin(set(train.text.map(normalized))).any()
external=external.drop_duplicates(subset=["model","normalized"],keep="first").reset_index(drop=True)
assert len(external)==8450, f"Unexpected external data snapshot: {len(external)}"
assert external.model.nunique()==18
print("restricted external",len(external),"rows")


In [ ]:
# Fold-independent final fit: exact V2 features + restricted external rows
EXTERNAL_WEIGHT=0.10
TYPO_WEIGHT=0.05   # selected via prompt-grouped five-fold OOF BA=0.400062
xtrain=train.text.tolist()
xtest=test.text.tolist()
combined=xtrain+external.text.tolist()
targets=np.concatenate([train.model.to_numpy(),external.model.to_numpy()])
weights=np.concatenate([
    np.ones(len(train),dtype=np.float32),
    np.full(len(external),EXTERNAL_WEIGHT,dtype=np.float32)
])
char=TfidfVectorizer(analyzer="char",ngram_range=(3,5),min_df=2,max_features=100000,
                     sublinear_tf=True,dtype=np.float32)
word=TfidfVectorizer(analyzer="word",ngram_range=(1,2),min_df=2,max_features=80000,
                     sublinear_tf=True,dtype=np.float32)
ct,cv=char.fit_transform(combined),char.transform(xtest)
wt,wv=word.fit_transform(combined),word.transform(xtest)
styles=style_matrix(combined)
scale=StandardScaler().fit(styles)
st=csr_matrix(np.clip(scale.transform(styles),-4,4))
sv=csr_matrix(np.clip(scale.transform(style_matrix(xtest)),-4,4))
X=hstack([ct,wt,st*0.08],format="csr")
XV=hstack([cv,wv,sv*0.08],format="csr")
model=LinearSVC(C=1.0,max_iter=4000)
model.fit(X,targets,sample_weight=weights)
scores=model.decision_function(XV)


In [ ]:
# Typography correction is estimated from OFFICIAL labeled train data only.
MARKS=["—","–","’","‘","“","”","…","•","  ","\n","**","##",";",":","?","!"]
def fingerprint(texts):
    return np.asarray([[float(m in text) for m in MARKS] for text in texts],dtype=np.float64)
def log_likelihood(xtr,ytr,xva,labels,alpha=2):
    binary_tr=fingerprint(xtr)
    binary_va=fingerprint(xva)
    global_p=(binary_tr.sum(axis=0)+alpha)/(len(xtr)+2*alpha)
    out=np.zeros((len(xva),len(labels)),dtype=np.float64)
    for col,lab in enumerate(labels):
        subset=binary_tr[ytr==lab]
        prob=(subset.sum(axis=0)+alpha)/(len(subset)+2*alpha)
        log_p=np.log(prob/global_p)
        log_not=np.log((1-prob)/(1-global_p))
        out[:,col]=(binary_va@log_p)+((1-binary_va)@log_not)
    return out

labels=model.classes_
assert len(labels)==18
typography=log_likelihood(xtrain,train.model.to_numpy(),xtest,labels)
pred=labels[np.argmax(scores+TYPO_WEIGHT*typography,axis=1)]
submission=pd.DataFrame({"id":test.id,"model":pred})
assert submission.id.equals(test.id)
assert submission.id.is_unique and submission.notna().all().all()
assert submission.model.isin(train.model.unique()).all()
dest=Path("/kaggle/working/submission.parquet")
submission.to_parquet(dest,index=False)
print("written",len(submission),"rows to",dest)
print(submission.head())
